# 🏐 Project Ē-Kóng (會講) v2 — Colab 啟動器

> **架構**：語音輸入（Groq Whisper API）× 文字輸出（純文字推播）
> **不需要 GPU** 也能跑（Groq 處理 STT，本地只跑 LLM）

## 必要 Secrets（左側 🔑 圖示加入）

| Secret 名稱 | 說明 |
|-------------|------|
| `LINE_CHANNEL_ACCESS_TOKEN` | LINE Channel Access Token |
| `LINE_CHANNEL_SECRET` | LINE Channel Secret |
| `NGROK_AUTH_TOKEN` | [ngrok](https://dashboard.ngrok.com/) Auth Token |
| `GROQ_API_KEY` | [console.groq.com](https://console.groq.com/) 免費 2000 分鐘/天 |

## 執行順序

| Cell | 說明 | 必要？ |
|------|------|:------:|
| 1 | 確認環境（GPU / Python） | ✅ |
| 2 | Clone / 更新程式碼 | ✅ |
| 3 | 建立 `.env` 設定檔 | ✅ 首次必跑 |
| 4 | 完整啟動（安裝依賴 + ngrok + FastAPI） | ✅ 首次必跑 |
| 5 | 快速重啟（跳過安裝，直接起服務） | 重啟 Colab 時用 |
| 6 | 測試 Regex 路由（不需模型） | 開發驗證用 |

In [ ]:
# ── Cell 1: 確認環境 ─────────────────────────────────────────────────────────
import subprocess, sys

# GPU 狀態
result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
gpu_info = result.stdout.strip()
if gpu_info:
    print(f'✅ GPU：{gpu_info}')
else:
    print('⚠️  未偵測到 GPU（v2 架構在 CPU-only 模式也可運行，但 LLM 推論會較慢）')

print(f'Python {sys.version}')

# 確認 Groq 套件可安裝（預先 check）
try:
    import groq
    print(f'✅ groq 套件已安裝：{groq.__version__}')
except ImportError:
    print('ℹ️  groq 套件尚未安裝，Cell 4 執行後會自動安裝')

In [ ]:
# ── Cell 2: Clone 或更新程式碼 ───────────────────────────────────────────────
import os
from pathlib import Path

REPO_URL = 'https://github.com/tingwei1231/E-Kong.git'  # ← 你的 repo URL
WORK_DIR = Path('/content/E-Kong')

if WORK_DIR.exists():
    print('🔄 更新程式碼...')
    ret = os.system(f'git -C {WORK_DIR} pull')
    print('✅ 已更新至最新版本' if ret == 0 else '⚠️  pull 失敗，請檢查網路')
else:
    print('📥 Clone 程式碼...')
    ret = os.system(f'git clone {REPO_URL} {WORK_DIR}')
    print('✅ Clone 完成' if ret == 0 else '❌ Clone 失敗，請確認 REPO_URL')

os.chdir(WORK_DIR)
print(f'📂 工作目錄：{os.getcwd()}')

In [ ]:
# ── Cell 3: 建立 .env 設定檔 ─────────────────────────────────────────────────
# 使用 Colab Secrets（左側 🔑）自動讀取敏感資訊
# Google Sheets CSV 網址請直接在下方填入
from google.colab import userdata

# ── 從 Colab Secrets 讀取（請先在左側 🔑 新增以下四個 Secret）────────────────
LINE_TOKEN  = userdata.get('LINE_CHANNEL_ACCESS_TOKEN')
LINE_SECRET = userdata.get('LINE_CHANNEL_SECRET')
NGROK_TOKEN = userdata.get('NGROK_AUTH_TOKEN')
GROQ_KEY    = userdata.get('GROQ_API_KEY')  # ← STT 必填，去 console.groq.com 申請

# ── LLM 模型路徑（改成你的實際路徑）───────────────────────────────────────────
# 推薦模型（擇一下載放到 Google Drive）：
#   Qwen2.5-1.5B-Instruct-Q4_K_M.gguf  ← 輕量，CPU 也能跑
#   Qwen2.5-7B-Instruct-Q4_K_M.gguf   ← 效果較好，建議 T4 GPU
LLM_MODEL_PATH = '/content/drive/MyDrive/ekong_models/qwen2.5-1.5b-instruct-q4_k_m.gguf'

# ── Google Sheets CSV 公開網址（在 Sheets → 檔案 → 發布到網路 → CSV 格式）────
# 每個分頁都要分別發布，填入對應網址
CSV_SCORE             = ''  # 比分分頁
CSV_GROUPS            = ''  # 分組名單分頁
CSV_STANDINGS         = ''  # 積分表分頁
CSV_LOSER_STANDINGS   = ''  # 敗者組積分分頁
CSV_ELIMINATION       = ''  # 晉淘結果分頁

# ─────────────────────────────────────────────────────────────────────────────
# 驗證必填變數
missing = [k for k, v in {
    'LINE_CHANNEL_ACCESS_TOKEN': LINE_TOKEN,
    'LINE_CHANNEL_SECRET':       LINE_SECRET,
    'NGROK_AUTH_TOKEN':          NGROK_TOKEN,
    'GROQ_API_KEY':              GROQ_KEY,
}.items() if not v]

if missing:
    print(f'❌ 以下 Secret 未設定：{missing}')
    print('   → 點左側 🔑 圖示新增後重跑此 Cell')
else:
    env_content = f"""LINE_CHANNEL_ACCESS_TOKEN={LINE_TOKEN}
LINE_CHANNEL_SECRET={LINE_SECRET}
NGROK_AUTH_TOKEN={NGROK_TOKEN}
GROQ_API_KEY={GROQ_KEY}
APP_PORT=8000
LOG_LEVEL=INFO
LLM_MODEL_PATH={LLM_MODEL_PATH}
LLM_N_GPU_LAYERS=35
LLM_MAX_TOKENS=512
LLM_TEMPERATURE=0.7
GOOGLE_SHEET_CSV_SCORE={CSV_SCORE}
GOOGLE_SHEET_CSV_GROUPS={CSV_GROUPS}
GOOGLE_SHEET_CSV_STANDINGS={CSV_STANDINGS}
GOOGLE_SHEET_CSV_LOSER_STANDINGS={CSV_LOSER_STANDINGS}
GOOGLE_SHEET_CSV_ELIMINATION={CSV_ELIMINATION}
"""
    with open('/content/E-Kong/.env', 'w') as f:
        f.write(env_content)
    print('✅ .env 已建立，包含以下欄位：')
    for line in env_content.strip().split('\n'):
        key = line.split('=')[0]
        val = line.split('=', 1)[1] if '=' in line else ''
        # 敏感資訊遮罩
        display = (val[:6] + '…') if val and any(k in key for k in ['TOKEN','SECRET','KEY']) else (val or '（未填）')
        print(f'   {key} = {display}')

In [ ]:
# ── Cell 4: 完整啟動（首次執行，安裝依賴 + ngrok + FastAPI）─────────────────
# 首次約需 1–3 分鐘安裝套件（groq 輕量，比原本快很多）
# 若已安裝過依賴，下次重啟 Colab 直接跑 Cell 5 即可
%cd /content/E-Kong
!python setup_colab.py

In [ ]:
# ── Cell 5: 快速重啟（已安裝依賴，跳過安裝直接起服務）──────────────────────
import os, subprocess, sys, time
from dotenv import load_dotenv
from pyngrok import conf as ngrok_conf, ngrok
import httpx

os.chdir('/content/E-Kong')
load_dotenv(override=True)

APP_PORT = int(os.getenv('APP_PORT', '8000'))

# ── 清理舊 tunnel ─────────────────────────────────────────────────────────
ngrok_conf.get_default().auth_token = os.environ['NGROK_AUTH_TOKEN']
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
    ngrok.kill()
except Exception:
    pass
os.system('pkill -f uvicorn 2>/dev/null'); time.sleep(1)

# ── 啟動 ngrok ────────────────────────────────────────────────────────────
tunnel = ngrok.connect(f'127.0.0.1:{APP_PORT}', bind_tls=True)
public_url = tunnel.public_url.rstrip('/')
print(f'🌐 ngrok URL: {public_url}')

# ── 更新 LINE Webhook ─────────────────────────────────────────────────────
with httpx.Client(timeout=15) as client:
    resp = client.put(
        'https://api.line.me/v2/bot/channel/webhook/endpoint',
        headers={
            'Authorization': f"Bearer {os.environ['LINE_CHANNEL_ACCESS_TOKEN']}",
            'Content-Type': 'application/json',
        },
        json={'endpoint': f'{public_url}/webhook'},
    )
status = '✅' if resp.status_code == 200 else f'❌ [{resp.status_code}]'
print(f'{status} LINE Webhook 更新完成')

# ── 啟動 FastAPI ──────────────────────────────────────────────────────────
proc = subprocess.Popen(
    [sys.executable, '-m', 'uvicorn', 'app.main:app',
     '--host', '0.0.0.0', '--port', str(APP_PORT), '--log-level', 'info'],
    cwd='/content/E-Kong'
)
time.sleep(4)

# ── Smoke test ────────────────────────────────────────────────────────────
try:
    with httpx.Client(timeout=10) as client:
        h = client.get(f'{public_url}/health')
    data = h.json()
    print(f'✅ /health OK | LLM={data["models"].get("llm")} | STT={data["models"].get("stt_groq")}')
except Exception as e:
    print(f'⚠️  /health 尚未回應，稍等幾秒再試：{e}')

print(f'''
🎉 E-Kong v2 已啟動！
   Webhook  : {public_url}/webhook
   Health   : {public_url}/health
   Swagger  : {public_url}/docs
''')

In [ ]:
# ── Cell 6: 測試 Regex 路由（不需模型，純本地驗證）─────────────────────────
# 可在啟動前先驗證意圖分類邏輯是否正確
import os; os.chdir('/content/E-Kong')
import sys; sys.path.insert(0, '/content/E-Kong')

# 不需要 .env，直接測試 Regex
os.environ.setdefault('LINE_CHANNEL_ACCESS_TOKEN', 'dummy')
os.environ.setdefault('LINE_CHANNEL_SECRET', 'dummy')
os.environ.setdefault('NGROK_AUTH_TOKEN', 'dummy')

from app.services.agent import fast_intent_router

test_cases = [
    ('幫我把第一場比分改掉',          'Reject_Update   ← 應攔截'),
    ('更新比分：台大 25 政大 18',      'Reject_Update   ← 應攔截'),
    ('廣播：請108號選手到報到台',      'Broadcast'),
    ('A組第一場現在比分幾比幾',        'Query_Score_Status'),
    ('誰贏了？',                      'Query_Score_Status'),
    ('報名截止時間是幾點',             'Query_Schedule'),
    ('網高是幾公分',                   'Query_Schedule'),
    ('積分怎麼算',                     'Query_Schedule'),
    ('廁所在哪裡',                     'General_Chat'),
    ('什麼時候頒獎',                   'General_Chat'),
]

print(f'{"輸入":<30} {"預期":<25} {"實際":<25} 結果')
print('-' * 90)
all_pass = True
for text, expect_hint in test_cases:
    result = fast_intent_router(text)
    actual = result['intent'].value
    expect_intent = expect_hint.split()[0]
    ok = '✅' if actual == expect_intent else '❌'
    if actual != expect_intent:
        all_pass = False
    print(f'{text:<30} {expect_hint:<25} {actual:<25} {ok}')

print()
print('🎉 全部通過！' if all_pass else '⚠️  有測試案例未通過，請檢查 fast_intent_router 的 Regex pattern')